# 02c｜承接 seed 权重，在扩充通用语料上训练新阶段

130U-FinLLM｜130U 金融大语言模型

先完成 01b 并确认三部分备份校验成功。本页要求 seed_v1 至少 300 步的正式预训练检查点，
冻结其身份并沿用原 8K tokenizer。因为训练语料变化，采用明确的权重承接：优化器、调度器状态、
数据游标及阶段计数重新建立。原 seed 检查点不覆盖；这不是同一数据流的精确断点续跑。

默认先验证新阶段 0→1 步，再至新阶段累计 610 步，名义新增约 10M token 位置；
实际有效 tokens 由报告给出，母模型约 4.91M tokens 单独列出。学习率调度总跨度为 6104 步，
本段达到 610 步不表示 100M 全预算或金融大语言模型已训练完成。

选择「运行时 → 更改运行时类型 → T4 GPU」，按顺序运行。
重复同版本使用严格恢复；中断先等连接恢复、确认训练已停止，再从第一格重新执行。
不要同时启动第二个训练进程。更改数据、词表或运行语义时选择独立实验版本。

In [ ]:
import os, sys, json, shutil, subprocess, importlib
from pathlib import Path

REPOSITORY = "https://github.com/130U/130U-FinLLM.git"
REF = "main"  # 首次解析并显示提交；复现时替换成完整的 40 位提交 SHA。
PROJECT = Path("/content/130U-FinLLM")

def command(*args, cwd=None):
    # Explicitly relay subprocess output into the notebook's captured stdout.
    with subprocess.Popen([str(x) for x in args], cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, process.args)

if len(REF) == 40 and all(c in "0123456789abcdefABCDEF" for c in REF):
    REVISION = REF.lower()
else:
    refs = subprocess.check_output(["git", "ls-remote", REPOSITORY, REF], text=True).strip().splitlines()
    if len(refs) != 1:
        raise RuntimeError("REF 不能唯一解析；请填写完整提交 SHA。")
    REVISION = refs[0].split()[0]

if not PROJECT.exists():
    command("git", "clone", "--no-checkout", REPOSITORY, PROJECT)
else:
    remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=PROJECT, text=True).strip()
    if remote.rstrip("/").removesuffix(".git") != REPOSITORY.removesuffix(".git"):
        raise RuntimeError("/content/130U-FinLLM 已属于其他仓库，请选择新目录。")
command("git", "fetch", "--depth", "1", "origin", REVISION, cwd=PROJECT)
command("git", "checkout", "--detach", REVISION, cwd=PROJECT)
os.chdir(PROJECT)
print("本次固定代码提交：", REVISION)

# 保留 Colab 自带的 CUDA / PyTorch；此处仅安装项目的其他依赖。
import torch
from importlib import metadata
from packaging.requirements import Requirement

def tokenizer_install_requirements():
    # Colab can include Transformers even though this project does not use it.
    # Pass its active tokenizer bounds to pip so an old runtime can retain 0.22,
    # while a current runtime (or an already broken one) can use 0.23.
    requirements = ["tokenizers>=0.21,<0.24"]
    try:
        existing = metadata.requires("transformers") or []
    except metadata.PackageNotFoundError:
        existing = []
    for value in existing:
        requirement = Requirement(value)
        if requirement.name.lower().replace("_", "-") == "tokenizers":
            if requirement.marker is None or requirement.marker.evaluate({"extra": ""}):
                requirements.append(str(requirement))
    return list(dict.fromkeys(requirements))

def check_tokenizer_compatibility(requirements):
    installed = metadata.version("tokenizers")
    incompatible = [value for value in requirements
                    if installed not in Requirement(value).specifier]
    if incompatible:
        raise RuntimeError(f"tokenizers {installed} 与现有环境不兼容：{incompatible}")
    print("tokenizers 兼容性检查通过：", installed)

def check_project_dependencies():
    # Read the refreshed installed metadata, not the previous editable checkout.
    problems = []
    for value in metadata.requires("finllm-130u") or []:
        requirement = Requirement(value)
        if requirement.marker is not None and not requirement.marker.evaluate({"extra": ""}):
            continue
        try:
            installed = metadata.version(requirement.name)
        except metadata.PackageNotFoundError:
            problems.append(f"缺少 {requirement.name}")
            continue
        if installed not in requirement.specifier:
            problems.append(f"{requirement.name}=={installed} 不满足 {requirement}")
    if problems:
        raise RuntimeError("项目依赖检查失败：" + "; ".join(problems))
    print("项目已安装依赖检查通过。")

def activate_project_in_kernel():
    # pip installs the editable .pth for NEW Python processes. A notebook kernel
    # that was already running does not process that .pth again automatically.
    source = (PROJECT / "src").resolve()
    expected = source / "finllm"
    if not (expected / "__init__.py").is_file():
        raise RuntimeError(f"缺少项目 Python 源码：{expected}")
    sys.path[:] = [str(source)] + [entry for entry in sys.path if entry != str(source)]
    # Git may have changed code in the same directory since an earlier cell run.
    # Refresh only our modules; never reload Torch or its live CUDA state.
    for name in tuple(sys.modules):
        if name == "finllm" or name.startswith("finllm."):
            del sys.modules[name]
    importlib.invalidate_caches()
    package = importlib.import_module("finllm")
    runtime = importlib.import_module("finllm.runtime")
    for module, filename in ((package, "__init__.py"), (runtime, "runtime.py")):
        if Path(module.__file__).resolve() != expected / filename:
            raise RuntimeError(f"当前 Notebook 导入了错误的项目位置：{module.__file__}")
    if not callable(runtime.restore_seed_checkpoint):
        raise RuntimeError("当前 Notebook 的检查点恢复接口不可用。")
    print("当前 Notebook 内核导入检查通过：", runtime.__file__)

def check_fresh_process_import():
    # Verify the installed editable package separately, without injecting src
    # into PYTHONPATH. Both the current kernel and future CLI processes must work.
    probe = """import json, sys
from pathlib import Path
import finllm
import finllm.runtime as runtime
expected = Path(sys.argv[1]).resolve() / 'finllm'
for module, filename in ((finllm, '__init__.py'), (runtime, 'runtime.py')):
    if Path(module.__file__).resolve() != expected / filename:
        raise RuntimeError('Fresh process imported the wrong project: ' + str(module.__file__))
if not callable(runtime.restore_seed_checkpoint):
    raise RuntimeError('Fresh process checkpoint restore is unavailable')
print(json.dumps({'finllm': str(Path(finllm.__file__).resolve()), 'runtime': str(Path(runtime.__file__).resolve())}))
"""
    result = subprocess.check_output([sys.executable, "-c", probe, str(PROJECT / "src")],
                                     cwd=PROJECT, text=True)
    print("新 Python 进程导入检查通过：", result.strip())

if tuple(int(v) for v in torch.__version__.split("+")[0].split(".")[:2]) < (2, 5):
    raise RuntimeError("需要 PyTorch >= 2.5；请更换较新的 Colab 运行环境。")
tokenizer_requirements = tokenizer_install_requirements()
print("本次 tokenizers 约束：", tokenizer_requirements)
torch_version_before = metadata.version("torch")
torch_file_before = Path(torch.__file__).resolve()
# One resolver transaction replaces the previous finllm metadata AND its
# dependencies. Upgrading tokenizers first leaves old finllm bounds active and
# produces a misleading conflict warning even when the eventual install works.
# The exact installed Torch build is constrained, so pip keeps Colab's CUDA build.
command(sys.executable, "-m", "pip", "install", "-q", "-e", ".",
        "torch==" + torch_version_before, *tokenizer_requirements)
importlib.invalidate_caches()
if metadata.version("torch") != torch_version_before or Path(torch.__file__).resolve() != torch_file_before:
    raise RuntimeError("PyTorch 环境意外变化，请保留输出并停止训练。")
check_tokenizer_compatibility(tokenizer_requirements)
check_project_dependencies()
activate_project_in_kernel()
check_fresh_process_import()
print("Notebook 环境准备完成；PyTorch 保持：", torch_version_before)

def finllm(*args):
    command(sys.executable, "-u", "-m", "finllm", *args, cwd=PROJECT)

def require(*paths):
    missing = [str(p) for p in paths if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("缺少阶段输入，请先准备或从 Drive 恢复：\n" + "\n".join(missing))

finllm("environment")
disk = shutil.disk_usage("/content")
print(f"/content 剩余磁盘：{disk.free / 2**30:.2f} GiB")
Path("runtime_revision.txt").write_text(REVISION + "\n")

In [ ]:
MOUNT_DRIVE = True
if not MOUNT_DRIVE:
    raise RuntimeError("本阶段需要已有种子输入及版本化备份，请完成 Drive 挂载。")
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/130U-FinLLM")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive 项目目录：", DRIVE_ROOT)

In [ ]:
EXPANSION_RETENTION_BEFORE_SUCCEEDED = False
EXPANSION_PROBE_SUCCEEDED = False
EXPANSION_TRAIN_SUCCEEDED = False
EXPANSION_RETENTION_AFTER_SUCCEEDED = False
EXPANSION_TRAIN_BACKED_UP = False
SEED_VERSION = "seed_v1"
EXPANSION_VERSION = "expansion_v1"
MIN_PARENT_UPDATES = 300
PROBE_UPDATES = 1
TARGET_UPDATES = 610
if not torch.cuda.is_available():
    raise RuntimeError("请选择 T4 GPU 运行时；正式扩充阶段不在 CPU 上启动。")
from finllm.expansion_runtime import prepare_expansion_training, run_expansion_segment, retention_check, backup_expansion_training
expansion = prepare_expansion_training(
    PROJECT, DRIVE_ROOT, EXPANSION_VERSION, SEED_VERSION,
    min_parent_updates=MIN_PARENT_UPDATES, probe_updates=PROBE_UPDATES,
    target_updates=TARGET_UPDATES)
print("已恢复新阶段更新步数：", expansion["restored_updates"])
print("已冻结母检查点身份：", json.dumps(expansion["parent"], indent=2, ensure_ascii=False))
print("本次新阶段配置：", json.dumps(expansion["config"], indent=2, ensure_ascii=False))
print("GPU:", torch.cuda.get_device_name(0))

先测冻结母检查点在原 seed 通用验证分片上的限量 loss；训练后在完全相同的位置比较。
同时保留扩充语料自己的验证损失。两者都是文本预测诊断，均不是金融立场评分。
已有输入身份完全相同的测量时读回结果。

In [ ]:
EXPANSION_RETENTION_BEFORE_SUCCEEDED = False
EXPANSION_PROBE_SUCCEEDED = False
EXPANSION_TRAIN_SUCCEEDED = False
EXPANSION_RETENTION_AFTER_SUCCEEDED = False
EXPANSION_TRAIN_BACKED_UP = False
retention_before = retention_check(expansion, "before")
print(json.dumps(retention_before, indent=2, ensure_ascii=False))
EXPANSION_RETENTION_BEFORE_SUCCEEDED = True

In [ ]:
EXPANSION_PROBE_SUCCEEDED = False
EXPANSION_TRAIN_SUCCEEDED = False
EXPANSION_RETENTION_AFTER_SUCCEEDED = False
EXPANSION_TRAIN_BACKED_UP = False
if not globals().get("EXPANSION_RETENTION_BEFORE_SUCCEEDED", False):
    raise RuntimeError("请先成功完成母检查点的原验证集测量。")
expansion_probe = run_expansion_segment(expansion, "probe", finllm)
print(json.dumps(expansion_probe, indent=2, ensure_ascii=False))
EXPANSION_PROBE_SUCCEEDED = True

0→1 步验证成功后才训练到新阶段累计 610 步。已有更靠后的同版本检查点时只校验恢复进度，不回退。
每 10 步保存并同步 Drive；各段报告的 updates/tokens 属于新阶段累计计数，耗时只属于本次调用。
数据／tokenizer／配置／运行时不兼容会停止；不要取消 provenance 检查或改成随机初始化绕过错误。

In [ ]:
EXPANSION_TRAIN_SUCCEEDED = False
EXPANSION_RETENTION_AFTER_SUCCEEDED = False
EXPANSION_TRAIN_BACKED_UP = False
if not globals().get("EXPANSION_RETENTION_BEFORE_SUCCEEDED", False) or not globals().get("EXPANSION_PROBE_SUCCEEDED", False):
    raise RuntimeError("请先完成原验证集测量及单步权重承接验证。")
expansion_result = run_expansion_segment(expansion, "continue", finllm)
print(json.dumps(expansion_result, indent=2, ensure_ascii=False))
EXPANSION_TRAIN_SUCCEEDED = True

In [ ]:
EXPANSION_RETENTION_AFTER_SUCCEEDED = False
EXPANSION_TRAIN_BACKED_UP = False
if not globals().get("EXPANSION_TRAIN_SUCCEEDED", False):
    raise RuntimeError("新阶段训练尚未成功结束；保留已有备份。")
retention_after = retention_check(expansion, "after")
print(json.dumps(retention_after, indent=2, ensure_ascii=False))
EXPANSION_RETENTION_AFTER_SUCCEEDED = True

In [ ]:
EXPANSION_TRAIN_BACKED_UP = False
if not all(globals().get(flag, False) for flag in (
    "EXPANSION_RETENTION_BEFORE_SUCCEEDED", "EXPANSION_PROBE_SUCCEEDED",
    "EXPANSION_TRAIN_SUCCEEDED", "EXPANSION_RETENTION_AFTER_SUCCEEDED")):
    raise RuntimeError("测量／单步验证／训练尚未全部成功；不得显示完整阶段备份。")
expansion_train_backup = backup_expansion_training(expansion)
print(json.dumps(expansion_train_backup, indent=2, ensure_ascii=False))
shutil.copy2(PROJECT / "runtime_revision.txt", expansion["paths"]["drive"] / "runtime_revision.txt")
EXPANSION_TRAIN_BACKED_UP = True
print("扩充通用训练与校验备份完成：", expansion["paths"]["drive"])
print("原 seed 数据与 300 步成果保留：", expansion["paths"]["seed_drive"])

验收时保留单步与正式段报告、retention before/after、实际 token 数和 `verified: true` 最终备份输出。
新阶段模型参数和词表不变；原 seed 300 步与此页新阶段步数分别记录。
本页只训练扩充通用文本，不自动进入金融领域训练、SFT 或 DPO。
金融扩充分片已由 01b 准备，下一阶段还需匹配版本的混合配置和入口；人工资产／立场标签仍待确认。